In [ ]:
import pandas as pd
import tabulate

calls_dfd = {}
"""for k in range(1, 5):
    data_filename = f"calls_a{k}.csv.bz2"
    print(f"Chargement du fichier {data_filename}")
    calls_dfd[f"a{k}"] = pd.read_csv(
        data_filename,
        parse_dates=[f"date_a{k_bis}" for k_bis in range(1, k + 1)],
    )"""
data_filename = "calls_a1.csv.bz2"
print(f"Chargement du fichier {data_filename}")
calls_dfd[1] = pd.read_csv(
    data_filename,
    parse_dates=["date_a1"]
)


#calls_dfd[1] = calls_dfd[1].head(3000000)

In [ ]:
print(tabulate.tabulate(calls_dfd[1].head(100), tablefmt="orgtbl", headers="keys", showindex=False))

In [ ]:
#On change le type des variables catégorielles

var_cat = ["weekday_a1", "hour_a1","caller_id_a1", "lead_answered_a1"]
for var in var_cat:
    calls_dfd[1][var] = calls_dfd[1][var].astype('category')

calls_dfd[1].info()

In [ ]:
###### Séparations des données d'apprentissage et de test ######
from sklearn.model_selection import train_test_split

train_calls, test_calls = train_test_split(calls_dfd[1], test_size=0.2, random_state=42)

"""separation = len(calls_dfd[1]) * 0.8
separation = int(separation)

train_calls = calls_dfd[1].head(separation)
test_calls = calls_dfd[1].tail(len(calls_dfd[1]) - separation)
"""

print(tabulate.tabulate(train_calls.head(100), tablefmt="orgtbl", headers="keys", showindex=False))

In [ ]:
print(tabulate.tabulate(test_calls.head(100), tablefmt="orgtbl", headers="keys", showindex=False))

In [ ]:
###### Création du modèle ######


#On crée les variables du RB et on les labelise

import pyAgrum as gum
import pyAgrum.lib.notebook as gnb

#var_to_model = ["hour_a1", "hour_a2", "lead_answered_a2"]
var_to_model = ["weekday_a1", "hour_a1","caller_id_a1", "lead_answered_a1"]

var_bn = {}
for var in var_to_model:
    nb_values = len(train_calls[var].cat.categories)
    var_bn[var] = gum.LabelizedVariable(var, var, nb_values)

In [ ]:
#Changement des modalités des variables pour être fidèle aux données
        
for var in var_bn:
    existing_labels = set(var_bn[var].labels())
    for i, modalite in enumerate(calls_dfd[1][var].cat.categories):
        new_label = str(modalite)
        if new_label not in existing_labels:
            var_bn[var].changeLabel(i, new_label)
            existing_labels.add(new_label)

In [ ]:
#Ajout des variables

bn = gum.BayesNet("Modèle 1 pour les premiers appels")

for var in var_bn.values():
    bn.add(var)

print(bn)

In [ ]:
#Création des arcs

bn.addArc("weekday_a1", "lead_answered_a1")
bn.addArc("hour_a1", "lead_answered_a1")
bn.addArc("caller_id_a1", "lead_answered_a1")

bn

In [ ]:
#Apprentissage des paramètres

learner = gum.BNLearner(train_calls[var_to_model])
learner.useSmoothingPrior(1e-2)
learner.fitParameters(bn)

In [ ]:
#Afichage des LPC

bn.cpt("weekday_a1")

In [ ]:
bn.cpt("hour_a1")

In [ ]:
bn.cpt("caller_id_a1")

In [ ]:
#bn.cpt("lead_answered_a1")

In [ ]:
import numpy as np
import sys

def predict(bn, data, var_target, show_progress=False):
    """
    This function is used to predict the posterior probability of a target variable from observations  
    using a bayesian network model. 

    Inputs:
    - =bn=: the predictive model given as a =pyAgrum.BayesNet= object
    - =data=: the data containing the observations used to predict the target variable 
    as a =pandas.DataFrame= object
    - =var_target=: the name of the target variable as a =str= object

    Returns:
    - a =DataFrame= containing the posterior probability distribution of the target 
    variable given each observation in =data=.
    """
    # Initialize the inference engine
    inf_bn = gum.LazyPropagation(bn)
    inf_bn.setTargets({var_target})
    nb_data = len(data)
    target_size = bn.variable(var_target).domainSize()
    target_dom = np.array([bn.variable(var_target).label(i)
                           for i in range(target_size)])
    data_records = data.to_dict("records")
    post_prob = np.zeros((nb_data, target_size))
    for i in range(nb_data):
        # Set the evidence
        inf_bn.setEvidence(data_records[i])
        # Run inference
        inf_bn.makeInference()
        # Compute posterior probability of target variable
        post_prob[i, :] = inf_bn.posterior(var_target).toarray()
        # Erase evidence
        inf_bn.eraseAllEvidence()
        if show_progress:
            sys.stdout.write("predict progress: {0:3.0%}\r".format(i/nb_data))

    post_prob_df = pd.DataFrame(post_prob,
                                index=data.index,
                                columns=bn.variable(var_target).labels())
    post_prob_df.columns.name = var_target
    return post_prob_df


pred_prob = predict(bn, data=calls_dfd[1].loc[:10000, ["weekday_a1", "hour_a1","caller_id_a1", "lead_answered_a1"]],
                    var_target="lead_answered_a1",
                    show_progress=True)

pred_var = pred_prob.idxmax(axis=1).rename("pred")

In [ ]:
# Evaluation du taux de prediction du modèle

(calls_dfd[1].loc[:10000, "lead_answered_a1"] == pred_var).mean()

In [ ]:
pd.crosstab(calls_dfd[1].loc[:10000, "lead_answered_a1"], pred_var)

In [ ]:
#Analyse des numéros qui ont décroché à la première tentative. 

calls_dfd["a1"]["lead_answered_a1"].value_counts()